# Q8 data prep: Census block relationship file → CSV

**Input:** `tab2010_tab2020_st27_mn.txt`, the U.S. Census Bureau's *2020 Census Tabulation Block to 2010 Census Tabulation Block Relationship File, Minnesota* (pipe-delimited text).

**Outputs** (written next to the input file):

| File | What it is |
|---|---|
| `tab2010_tab2020_st27_mn.csv` | The same file as a plain CSV, every column kept as text so codes like `001` keep their leading zeros |
| `mn_tract_crosswalk_2010_2020.csv` | The file rolled up to 2010 tract → 2020 tract pairs, with land-area shares. This is what Q8 uses to move the 2019 USDA LILA flags (2010 tracts) onto 2020 tracts |

Run the cells top to bottom. Only the path in the next cell needs changing.

## 1. Settings

In [2]:
# Path to the input file.
# Local Jupyter: put the notebook in the same folder as the .txt, or give the full path.
# Google Colab: run the Drive cell below first, then point this at the file in Drive, e.g.
#   '/content/drive/MyDrive/MinneMUDAC 26 for BU/VG/Data for Q8/tab2010_tab2020_st27_mn.txt'
INPUT_TXT = '/content/drive/MyDrive/MinneMUDAC 26 for BU/VG/Data for Q8/Block Relationship/tab2010_tab2020_st27_mn.txt'

# Drop tract pieces smaller than this share of BOTH tracts (boundary slivers).
SLIVER_SHARE = 0.001   # 0.1%

In [1]:
# Google Colab only: mount Drive. Skip this cell in local Jupyter.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; skipping Drive mount.')

Mounted at /content/drive


## 2. Read the pipe-delimited file

In [3]:
import os
import pandas as pd

# dtype=str keeps leading zeros in state, county, tract and block codes.
# 'utf-8-sig' strips the byte-order mark at the start of the Census file.
raw = pd.read_csv(INPUT_TXT, sep='|', dtype=str, encoding='utf-8-sig')

print(f'{len(raw):,} rows × {raw.shape[1]} columns')
raw.head()

296,578 rows × 18 columns


,STATE_2010,COUNTY_2010,TRACT_2010,BLK_2010,BLKSF_2010,AREALAND_2010,AREAWATER_2010,BLOCK_PART_FLAG_O,STATE_2020,COUNTY_2020,TRACT_2020,BLK_2020,BLKSF_2020,AREALAND_2020,AREAWATER_2020,BLOCK_PART_FLAG_R,AREALAND_INT,AREAWATER_INT
0,19,063,070200,2010,NaN,478900,0,p,27,091,790400,2142,NaN,7775,0,p,2398,0
1,19,063,070200,2012,NaN,14074,0,p,27,091,790400,1127,NaN,48506,0,p,1712,0
2,19,063,070200,2012,NaN,14074,0,p,27,091,790400,2142,NaN,7775,0,p,1952,0
3,27,001,770100,1000,NaN,2118481,0,p,27,001,770100,1000,NaN,759277,0,NaN,759277,0
4,27,001,770100,1000,NaN,2118481,0,p,27,001,770100,1001,NaN,1140195,0,NaN,1140195,0


## 3. Check the file

Expected: about 296,578 rows, 18 columns, and a handful of border rows whose 2010 block is in Iowa (19), South Dakota (46) or Wisconsin (55).

In [4]:
expected_cols = ['STATE_2010','COUNTY_2010','TRACT_2010','BLK_2010','BLKSF_2010',
                 'AREALAND_2010','AREAWATER_2010','BLOCK_PART_FLAG_O',
                 'STATE_2020','COUNTY_2020','TRACT_2020','BLK_2020','BLKSF_2020',
                 'AREALAND_2020','AREAWATER_2020','BLOCK_PART_FLAG_R',
                 'AREALAND_INT','AREAWATER_INT']
missing = [c for c in expected_cols if c not in raw.columns]
assert not missing, f'Missing columns: {missing}'

print('2010 state → 2020 state row counts:')
print(raw.groupby(['STATE_2010', 'STATE_2020']).size())

2010 state → 2020 state row counts:
STATE_2010  STATE_2020
19          27                 3
27          27            296569
46          27                 1
55          27                 5
dtype: int64


## 4. Save the full file as CSV

In [5]:
out_dir = os.path.dirname(os.path.abspath(INPUT_TXT))
block_csv = os.path.join(out_dir, 'tab2010_tab2020_st27_mn.csv')

raw.to_csv(block_csv, index=False)
print('Saved', block_csv, f'({os.path.getsize(block_csv)/1e6:.1f} MB)')

Saved /content/drive/MyDrive/MinneMUDAC 26 for BU/VG/Data for Q8/Block Relationship/tab2010_tab2020_st27_mn.csv (20.6 MB)


## 5. Build the tract crosswalk

Each row of the block file is the overlap between one 2010 block and one 2020 block. Adding up the overlapping land area (`AREALAND_INT`) by tract pair gives how much of each 2010 tract falls in each 2020 tract.

- Only Minnesota-to-Minnesota pairs are kept (border blocks from other states dropped).
- Pairs with no shared land are dropped, which also drops a few water-only tracts.
- Slivers under 0.1% of both tracts are dropped.
- Weights are by land area, because the file has no population figures.

In [6]:
df = raw[(raw.STATE_2010 == '27') & (raw.STATE_2020 == '27')].copy()
df['AREALAND_INT'] = pd.to_numeric(df['AREALAND_INT'])

# 11-digit tract GEOIDs: state (2) + county (3) + tract (6)
df['GEOID_TRACT_10'] = df.STATE_2010 + df.COUNTY_2010 + df.TRACT_2010
df['GEOID_TRACT_20'] = df.STATE_2020 + df.COUNTY_2020 + df.TRACT_2020

xw = (df.groupby(['GEOID_TRACT_10', 'GEOID_TRACT_20'], as_index=False)['AREALAND_INT']
        .sum()
        .rename(columns={'AREALAND_INT': 'arealand_part'}))

tot10 = xw.groupby('GEOID_TRACT_10')['arealand_part'].transform('sum')
tot20 = xw.groupby('GEOID_TRACT_20')['arealand_part'].transform('sum')
xw['share_of_tract10'] = (xw.arealand_part / tot10).where(tot10 > 0)
xw['share_of_tract20'] = (xw.arealand_part / tot20).where(tot20 > 0)

xw = xw[xw.arealand_part > 0]
xw = xw[(xw.share_of_tract10 >= SLIVER_SHARE) | (xw.share_of_tract20 >= SLIVER_SHARE)]
xw = xw.sort_values(['GEOID_TRACT_10', 'GEOID_TRACT_20']).reset_index(drop=True)

print(f'{len(xw):,} tract pairs')
print(f'{xw.GEOID_TRACT_10.nunique():,} 2010 tracts → {xw.GEOID_TRACT_20.nunique():,} 2020 tracts')
print(f'{(xw.groupby("GEOID_TRACT_10").size() == 1).sum():,} 2010 tracts map to exactly one 2020 tract')
xw.head()

2,074 tract pairs
1,336 2010 tracts → 1,502 2020 tracts
842 2010 tracts map to exactly one 2020 tract


,GEOID_TRACT_10,GEOID_TRACT_20,arealand_part,share_of_tract10,share_of_tract20
0,27001770100,27001770100,1187766666,0.999368,1.000000
1,27001770100,27001770300,748189,0.000630,0.008567
2,27001770200,27001770200,1196043065,1.000000,0.999988
3,27001770300,27001770300,86580571,0.999870,0.991433
4,27001770400,27001770401,244066724,0.204806,1.000000


## 6. Save the tract crosswalk

In [7]:
tract_csv = os.path.join(out_dir, 'mn_tract_crosswalk_2010_2020.csv')
xw.to_csv(tract_csv, index=False)
print('Saved', tract_csv, f'({os.path.getsize(tract_csv)/1e3:.0f} KB)')

Saved /content/drive/MyDrive/MinneMUDAC 26 for BU/VG/Data for Q8/Block Relationship/mn_tract_crosswalk_2010_2020.csv (128 KB)


## How to use the crosswalk in Q8

Always read it with `dtype=str` for the GEOID columns so leading zeros are kept.

```python
xw = pd.read_csv('mn_tract_crosswalk_2010_2020.csv',
                 dtype={'GEOID_TRACT_10': str, 'GEOID_TRACT_20': str})
```

- **Counts** in the 2019 USDA file (e.g. low-access population): multiply by `share_of_tract10`, then sum by `GEOID_TRACT_20`.
- **Yes/no flags** (e.g. `LILATracts_1And10`): flag a 2020 tract if more than half its land came from flagged 2010 tracts, i.e. sum `share_of_tract20` over flagged pieces and test `> 0.5`.